# Experiment 5.3: A Two-Mode Equalizer
Four selectable outputs: Original, Drum Enhance, Guitar Enhance and Drum + Guitar Enhance. Process seconds 20–30 of the same stereo mixture recording. Compare all modes and experiment with gains and cutoff. NumPy, SciPy, Matplotlib and IPython are required.

## Interactive zoom
Run the installation cell once, restart the kernel, then run from the widget cell onward. Use the magnifying-glass toolbar button and drag a rectangle to zoom; Home resets the view. Inspect 0–1500 Hz for bass and the cutoff transition, or a smaller treble interval such as 2000–3000 Hz. Your notebook viewer must support Jupyter widgets. Saved PNGs remain static.

In [1]:
%pip install ipympl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\tejas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
%matplotlib widget

In [3]:
import math
import cmath
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from IPython.display import Audio, display

script_dir = Path.cwd()
project_dir = script_dir.parent if script_dir.name.lower() == 'source code' else script_dir
plot_dir = project_dir / 'plots'
output_dir = project_dir / 'audio outputs'
plot_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)
fs, raw = wavfile.read(script_dir / 'mixture.wav')
if raw.dtype != np.int16:
    raise ValueError('Expected signed 16-bit PCM input.')
if len(raw) < 30*fs:
    raise ValueError('Input must contain at least 30 seconds.')
audio_selected = raw[20*fs:30*fs].astype(float)/32768.0
if audio_selected.ndim != 2 or audio_selected.shape[1] != 2:
    raise ValueError('Expected stereo audio.')


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\tejas\\Downloads\\lssp-midterm-final\\source code\\mixture.wav'

## 1. Complementary low- and high-frequency components
Use a 101-tap (100th-order) Hamming-windowed sinc FIR low-pass. Centered convolution compensates its 50-sample delay for this offline experiment. Zero padding affects the boundaries. Obtain the high component by subtraction: high = original − low. A causal real-time implementation would instead delay the dry path by 50 samples.

In [ ]:
def manual_hamming(length):
    """Symmetric Hamming window: 0.54 - 0.46*cos(2*pi*n/(L-1))."""
    if length <= 0:
        return np.array([], dtype=float)
    if length == 1:
        return np.ones(1)
    window = np.zeros(length)
    for n in range(length):
        window[n] = 0.54 - 0.46 * math.cos(2 * math.pi * n / (length-1))
    return window


def manual_hanning(length):
    """Symmetric Hann window: 0.5 - 0.5*cos(2*pi*n/(L-1))."""
    if length <= 0:
        return np.array([], dtype=float)
    if length == 1:
        return np.ones(1)
    window = np.zeros(length)
    for n in range(length):
        window[n] = 0.5 - 0.5 * math.cos(2 * math.pi * n / (length-1))
    return window

def manual_convolve(signal, impulse_response):
    """Full linear convolution by accumulating each delayed, scaled input."""
    result = np.zeros(len(signal) + len(impulse_response) - 1, dtype=float)
    for k, tap in enumerate(impulse_response):
        if tap != 0:
            # This slice performs result[n+k] += signal[n]*tap for every n.
            result[k:k+len(signal)] += signal * tap
    return result

N = 101

def split_bands(x, cutoff):
    if not 0 < cutoff < fs/2:
        raise ValueError('Cutoff must be between 0 and Nyquist.')
    k = np.arange(N) - (N-1)/2
    h = 2*cutoff/fs * np.sinc(2*cutoff/fs*k) * manual_hamming(N)
    h /= h.sum()
    delay = (N-1)//2
    low = np.column_stack([manual_convolve(x[:,ch], h)[delay:delay+len(x)] for ch in range(2)])
    high = x-low
    np.testing.assert_allclose(low+high, x, atol=1e-12)
    return low, high, h

cutoff = 500
low, high, h = split_bands(audio_selected, cutoff)


## 2. Four modes and multiplexer
Drum mode boosts bass; guitar mode boosts treble. The combined mode uses both selected gains. Unequal gains give the combined output a changed tonal balance; equal gains would simply amplify the original.

In [ ]:
drum_gain = 1.8
guitar_gain = 1.4

def make_modes(x, low, high, bass, treble):
    return {
        'Original': x.copy(),
        'Drum Enhance': bass*low + high,
        'Guitar Enhance': low + treble*high,
        'Drum + Guitar Enhance': bass*low + treble*high,
    }

modes = make_modes(audio_selected, low, high, drum_gain, guitar_gain)
mode = 'Drum + Guitar Enhance'  # Change to any key above.
if mode not in modes:
    raise ValueError('Unknown equalizer mode.')
output = modes[mode]


## 3. Listen to every mode and export
One shared attenuation factor protects all four exports from clipping and preserves their relative levels. Players do not independently normalize the signals. Listen to all four before writing listening observations.

In [ ]:
peak = max(np.max(np.abs(y)) for y in modes.values())
shared_gain = min(1., .99/max(peak, 1e-12))
for name, y in modes.items():
    print(name)
    display(Audio((y*shared_gain).T, rate=fs, normalize=False))
    filename = '5.3_' + name.lower().replace(' + ', '_and_').replace(' ', '_') + '.wav'
    wavfile.write(output_dir / filename, fs, (y*shared_gain).astype(np.float32))


## 4. Compare audio magnitude spectra
Use the same full ten-second segment, Hann window and amplitude reference for every mode. The one-sided spectra below describe the audio, rather than only the filter response. Analyze signals before export attenuation.

In [ ]:
def fft(signal):
    """Recursive radix-2 Cooley-Tukey FFT; pad to a power of two."""
    size = len(signal)
    if size == 0:
        return []
    if size == 1:
        return [complex(signal[0])]
    if size & (size-1):
        padded = 1 << (size-1).bit_length()
        signal = list(signal) + [0]*(padded-size)
        size = padded
    even = fft(signal[::2])
    odd = fft(signal[1::2])
    result = [0j]*size
    for k in range(size//2):
        twiddle = cmath.exp(-2j*cmath.pi*k/size)*odd[k]
        result[k] = even[k]+twiddle
        result[k+size//2] = even[k]-twiddle
    return result

def spectrum(x):
    window = manual_hanning(len(x))
    fft_length = 1 << (len(x)-1).bit_length()
    positive_length = fft_length//2+1
    X = np.column_stack([fft(x[:,ch]*window)[:positive_length] for ch in range(x.shape[1])])
    mag = np.abs(X)/window.sum()
    mag[1:-1] *= 2
    return np.arange(positive_length)*fs/fft_length, 20*np.log10(np.maximum(mag,1e-12))

fig, axes = plt.subplots(2,1,figsize=(13,8),sharex=True)
for name,y in modes.items():
    f, db = spectrum(y)
    for ch in range(2):
        axes[ch].plot(f, db[:,ch],label=name,alpha=.7)
for ch,ax in enumerate(axes):
    ax.set(title=['Left channel','Right channel'][ch],ylabel='Magnitude (dB re 1)',xlim=(0,min(8000,fs/2)))
    ax.legend(); ax.grid(True)
axes[-1].set_xlabel('Frequency (Hz)')
fig.tight_layout()
fig.savefig(plot_dir/'5.3_magnitude_spectra.png',dpi=150)
plt.show()


## 5. Controlled cutoff and gain experiments
Compare 250, 500 and 1000 Hz with gains fixed at 1.8/1.4. Then compare gain pairs 1.2/1.1, 1.8/1.4 and 2.2/1.6 with cutoff fixed at 500 Hz. The shared baseline makes five unique configurations. Each compares the original and all three processed modes, saves audio and plots, and provides players. The maximum-peak bound below uses band peaks, so every trial uses the same safe playback gain.

In [ ]:
trials = [(250,1.8,1.4),(500,1.8,1.4),(1000,1.8,1.4),
          (500,1.2,1.1),(500,2.2,1.6)]
splits = {fc: split_bands(audio_selected,fc) for fc,_,_ in trials}
bound = max(max(np.max(np.abs(audio_selected)),
                bass*np.max(np.abs(splits[fc][0])) + treble*np.max(np.abs(splits[fc][1])))
            for fc,bass,treble in trials)
trial_gain = min(1., .99/max(bound,1e-12))
original_f, original_db = spectrum(audio_selected)
for fc,bass,treble in trials:
    lo,hi,_ = splits[fc]
    trial_modes = make_modes(audio_selected,lo,hi,bass,treble)
    tag = f'cutoff_{fc}Hz_gL_{bass:g}_gH_{treble:g}'
    fig,axes = plt.subplots(2,1,figsize=(12,8),sharex=True)
    for name,y in trial_modes.items():
        print(tag, name)
        display(Audio((y*trial_gain).T,rate=fs,normalize=False))
        safe_name = name.lower().replace(' + ','_and_').replace(' ','_')
        wavfile.write(output_dir/f'5.3_{tag}_{safe_name}.wav',fs,(y*trial_gain).astype(np.float32))
        f,db = spectrum(y)
        for ch in range(2):
            # Spectral difference makes the tonal changes easier to compare.
            valid = original_db[:,ch] > original_db[:,ch].max()-70
            delta = np.where(valid, db[:,ch]-original_db[:,ch], np.nan)
            axes[ch].plot(f,delta,label=name,alpha=.7)
    for ch,ax in enumerate(axes):
        ax.set(title=f'{tag}, channel {ch+1}',ylabel='Change from original (dB)',xlim=(20,5000))
        ax.set_xscale('log'); ax.legend(); ax.grid(True)
    axes[-1].set_xlabel('Frequency (Hz)')
    fig.tight_layout()
    fig.savefig(plot_dir/f'5.3_{tag}_spectral_comparison.png',dpi=150)
    plt.show()


## Interpretation
Bass gain should increase low-frequency energy in Drum Enhance; treble gain should increase high-frequency energy in Guitar Enhance. The combined mode applies both. Raising the cutoff extends the low band to higher frequencies; higher gains increase the effect and required headroom. The 101-tap FIR has a broad transition, especially at low cutoffs, so these are gradual bands rather than a sharp split.

The plots and players support every required comparison. Record which cutoff and gain pair best separates drum body from string detail after listening; these expectations are not fabricated listening observations. Spectral-difference curves omit bins more than 70 dB below the original peak to avoid unstable ratios near silence. The code checks band reconstruction and uses shared gain for safe playback/export.